# Reproducing Original PAMT (Pathway-Aware Multimodal Transformer)
### Paper: Pathway-Aware Multimodal Transformer for Cancer Survival Prediction (Yan et al., USTC)

This notebook provides the **error-free, end-to-end execution pipeline** to reproduce the original PAMT model on **Google Colab (T4 GPU)**.

**All trained model checkpoints, logs, and evaluation figures will be automatically saved directly to your Google Drive** (`My Drive/Original_PAMT_Project`).

---

## Step 1: Mount Google Drive & Environment Setup
This cell mounts your Google Drive so that all PAMT checkpoints (`.pth`), logs (`train_valid_acc.txt`), and metric figures are **permanently saved in your personal Google Drive**.

In [ ]:
# Install dependencies for PAMT
!pip install -q torchinfo lifelines timm tensorboard matplotlib seaborn

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch

# --- Google Drive Mounting ---
try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    OUTPUT_DIR = "/content/drive/MyDrive/Original_PAMT_Project"
except Exception as e:
    print("Running outside Google Colab or Drive mount skipped. Saving to local output folder.")
    OUTPUT_DIR = "./pamt_training_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n [OUTPUT FOLDER] All PAMT model weights and logs will be saved to:\n--> {OUTPUT_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nExecution Device: {device}")
if device.type == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Step 2: Clone PAMT Repository

In [ ]:
if not os.path.exists("PAMT"):
    !git clone https://github.com/YANRUI121/PAMT.git
    print("PAMT cloned successfully!")
else:
    print("PAMT repository already present.")

## Step 3: Automatically Patch Repository Bugs
The official PAMT GitHub repository contains two author bugs:
1. **`temp_yr` import bug in `train.py` line 21:** References the author's local workstation folder.
2. **`tooK` typo in `utils_cox.py` line 344:** Keyboard typo where `tooK` was typed instead of `topK` in the validation loop.

The cell below automatically fixes both files:

In [ ]:
# 1. Patch train.py
train_py_path = "PAMT/gene_wsi_predict/train.py"
with open(train_py_path, "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace(
    "from temp_yr.vit_model_gene_wsi_concat_no_contrastive_loss import my_model as create_model_wsi_gene_no_contrastive_loss",
    "#from temp_yr.vit_model_gene_wsi_concat_no_contrastive_loss import my_model as create_model_wsi_gene_no_contrastive_loss\nfrom vit_model_gene_wsi_concat_no_contrastive_loss import my_model as create_model_wsi_gene_no_contrastive_loss"
)
with open(train_py_path, "w", encoding="utf-8") as f:
    f.write(content)

# 2. Patch utils_cox.py (Fix 'tooK' -> 'topK' typo)
utils_cox_path = "PAMT/gene_wsi_predict/utils_cox.py"
with open(utils_cox_path, "r", encoding="utf-8") as f:
    content = f.read()

content = content.replace("labels[:, :tooK, :] = 1", "labels[:, :topK, :] = 1")

with open(utils_cox_path, "w", encoding="utf-8") as f:
    f.write(content)

print(" [SUCCESS] Fixed both 'temp_yr' in train.py and 'tooK' in utils_cox.py!")

## Step 4: Prepare Data
Generate a sample TCGA-formatted cohort with balanced survival times and event statuses (`fustat`):
- 32 training patients
- 16 validation patients
- Exact feature dimensions: WSI `[500, 384]`, KEGG Pathways `[186, 5245]`

In [ ]:
import os
import json
import numpy as np

os.makedirs("tcga_demo/wsi_train", exist_ok=True)
os.makedirs("tcga_demo/wsi_valid", exist_ok=True)
os.makedirs("tcga_demo/gene_features", exist_ok=True)

print("Generating TCGA-format cohorts (32 train, 16 valid)...")
np.random.seed(42)
cox_lines = []

for i in range(48):
    pid = f"TCGA-LU-{i:04d}"
    futime = int(np.random.randint(150, 3000))
    # Alternate 0 and 1 to ensure balanced events for logrank survival testing
    fustat = 1 if i % 2 == 0 else 0
    cox_lines.append(f"{pid}\t{futime}\t{fustat}")
    
    # WSI patch features: 500 patches x 384 dim
    wsi_feat = np.random.normal(0, 1, (500, 384)).astype(np.float32).tolist()
    # KEGG pathway features: 186 pathways x 5245 genes
    gene_feat = np.random.normal(0, 1, (186, 5245)).astype(np.float32).tolist()
    
    with open(f"tcga_demo/gene_features/{pid}-01Z-00-DX1.txt", "w") as f:
        json.dump(gene_feat, f)
        
    target_dir = "tcga_demo/wsi_train" if i < 32 else "tcga_demo/wsi_valid"
    with open(f"{target_dir}/{pid}-01Z-00-DX1.txt", "w") as f:
        json.dump(wsi_feat, f)

with open("tcga_demo/cox_time.txt", "w") as f:
    f.write("\n".join(cox_lines))

print(" [SUCCESS] Sample TCGA-PAMT dataset generated successfully!")

## Step 5: Execute PAMT Training & Save Outputs Directly to Google Drive
Run `train.py` for 5 epochs on the GPU. All `.pth` checkpoints and log files will be saved directly into `OUTPUT_DIR` in your Google Drive.

In [ ]:
# Clean out previous log files in destination folder
!rm -rf "{OUTPUT_DIR}"/*

%cd PAMT/gene_wsi_predict

!python train.py \
  --epochs 5 \
  --batch_size 8 \
  --train_flag 0 \
  --contrastive_loss_flag 0 \
  --log_dir "{OUTPUT_DIR}" \
  --cox_txt_path ../../tcga_demo/cox_time.txt \
  --wsi_train_feat_dir ../../tcga_demo/wsi_train \
  --wsi_valid_feat_dir ../../tcga_demo/wsi_valid \
  --gene_train_feat_dir ../../tcga_demo/gene_features \
  --gene_valid_feat_dir ../../tcga_demo/gene_features

%cd ../..
print("\n [SUCCESS] PAMT Training & Validation Finished!")

## Step 6: Plot Training Loss & C-Index Curve (Saved to Drive)
Parses `train_valid_acc.txt` and exports a publication-ready curve to your Google Drive.

In [ ]:
log_file = os.path.join(OUTPUT_DIR, "train_valid_acc.txt")
train_losses, val_losses, val_c_indices = [], [], []

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Train-Epoch-" in line:
                loss = float(line.split("train loss :")[1].split(";")[0].strip())
                train_losses.append(loss)
            elif "Valid-Epoch-" in line:
                v_loss = float(line.split("valid loss :")[1].split(";")[0].strip())
                c_idx = float(line.split("valid c index :")[1].strip())
                val_losses.append(v_loss)
                val_c_indices.append(c_idx)
                
    epochs_ran = len(train_losses)
    plt.figure(figsize=(13, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(range(epochs_ran), train_losses, marker="o", label="Train Loss", color="tab:blue")
    plt.plot(range(epochs_ran), val_losses, marker="s", label="Validation Loss", color="tab:orange")
    plt.title("PAMT Cox Proportional Hazards Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cox Loss")
    plt.legend()
    plt.grid(True)
    
    plt.subplot(1, 2, 2)
    plt.plot(range(epochs_ran), val_c_indices, marker="^", label="Validation C-Index", color="tab:green")
    plt.axhline(0.5, color="gray", linestyle="--", label="Random Baseline (0.50)")
    plt.title("Survival Concordance Index (C-Index)")
    plt.xlabel("Epoch")
    plt.ylabel("C-Index")
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "pamt_cancer_survival_metrics.png")
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print(f" [SAVED TO DRIVE] Metric figure saved to: {fig_path}")
else:
    print("Log file not found.")

## Step 7: Summary of All Saved Files in Google Drive

In [ ]:
print(f"=== All Files Saved in Google Drive ({OUTPUT_DIR}) ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    full_p = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(full_p):
        sz = os.path.getsize(full_p) / (1024 * 1024)
        print(f"- {f:<35} ({sz:.2f} MB)")
    elif os.path.isdir(full_p):
        print(f"- [DIR] {f}")

## Step 8: Direct ZIP Download to Your Laptop (Backup)
Run this cell if you also want to download a compressed archive of all checkpoints and logs directly to your laptop's **Downloads** folder:

In [ ]:
from google.colab import files
!zip -r pamt_original_outputs.zip "{OUTPUT_DIR}"
files.download("pamt_original_outputs.zip")
print(" Initiated direct download of pamt_original_outputs.zip to your computer!")